# Занятие 2, пара 2. Титаник. Первые выводы

Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai

После пары: загрузить CSV, проверить пропуски, отфильтровать строки, сравнить группы и записать вывод с ограничением. Каждая строка — пассажир из учебной выборки, не весь список людей на корабле. Итог — выполненные задания и начало ЛР1.

### Подготовка

Запустите ячейку: она найдёт папку курса и подключит автопроверки.

In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check

PROJECT_DIR


In [ ]:
import pandas as pd
from IPython.display import display
df = pd.read_csv(DATA_DIR / "titanic.csv", index_col="PassengerId")
display(df.head())
print("Строк, столбцов:", df.shape)

## Что расскажет DataFrame

Сначала размер, несколько строк, типы и пропуски. `info()` печатает отчёт, а не возвращает таблицу: не записывайте его результат обратно в df.

In [ ]:
df.info()
display(df.isna().sum().sort_values(ascending=False))
display(df.describe())

`None` имеет тип NoneType. В обычном числовом столбце pandas пропуск часто представлен NaN, поэтому целые числа могут стать float. Есть и nullable-типы (например Int64); их рассмотрим при очистке. Строковые типы зависят от версии pandas.

Не удаляем строки только потому, что в Cabin много пропусков. Пропуск — характеристика данных, решение требует смысла.

In [ ]:
demo = pd.Series([10, 20, None])
print(type(None), demo.dtype)
display(demo)

## Фильтры и сортировка

Для нескольких условий используем скобки и `&`/`|`. `and` не объединяет массивы условий. Сначала выбираем строки, затем интересующие столбцы.

In [ ]:
adults = df.loc[(df["Age"] >= 18) & (df["Pclass"] == 3), ["Name", "Age", "Fare"]]
display(adults.sort_values("Fare", ascending=False).head())

### ✏️ Ваш ход 1 — паспорт выборки

Получите `shape_answer = df.shape`, `missing_column` — название столбца с наибольшим числом пропусков, `class_counts` — число пассажиров каждого класса (по порядку 1, 2, 3).

In [ ]:
# ✏️ ваш код здесь

In [ ]:
check('02.2.1', shape_answer, missing_column, class_counts)

## Среднее у нулей и единиц

Survived: 1 — выжил, 0 — нет. Среднее этого столбца равно доле выживших. `.groupby()` повторяет расчёт отдельно для каждой группы. Сегодня используем один простой пример; более сложные группировки будут позже.

In [ ]:
display(df.groupby("Pclass")["Survived"].agg(["count", "mean"]))

### ✏️ Ваш ход 2 — сравнение

Получите `survival_by_sex` — долю выживших по Sex, и `max_fare` — максимальную Fare. Под таблицей напишите, кого именно описывают эти доли и почему это не доказательство причины.

In [ ]:
# ✏️ ваш код здесь

In [ ]:
check('02.2.2', survival_by_sex, max_fare)

## Ваше небольшое исследование

Выберите вопрос: доля выживших по классу; возраст в разных классах; пассажиры с самыми дорогими билетами. Получите таблицу, подпишите единицы и запишите два вывода и одно ограничение. Укажите размер сравниваемых групп. Минимум — один вопрос; быстрым — сравнить пол внутри каждого класса.

In [ ]:
# Ваш код и ниже Markdown с выводами

## Пара слов о признаках

Когда мы изучаем датафрейм с целью анализа данных, то стоит понимать, какие признаки встречаются в столбцах. Их можно разделить на следующие группы:

1. Количественные
   1. Дискретные
   2. Непрерывные
2. Качественные (категориальные)
   1. Порядковые
   2. Номинальные

В чем их отличие?

Количественные признаки задаются числом. При этом дискретные задаются конкретными числами с неким шагом. Например, число студентов в аудитории мы можем описать только целыми неотрицательными числами. У нас не может сидеть полтора студента. А вот непрерывные признаки можно задать любым числом. Например, масса камня или скорость автомобиля.

Качественные признаки могут быть заданы текстово, либо числами, если отражают разбиение на категории. Например, наличие детей в семье, заданное числами 0 и 1 будет не количественным, а качественным признаком, хоть и указано числами. Порядковые признаки отражают некий порядок. Например, уровень образования: начальное, общее, среднее и т.д. А номинальные нельзя распределить по порядку. Примером этих признаков могут быть цвета воздушных шариков: красный. синий, зеленый...

А теперь найдите в таблице пассажиров количественный признак и категорию, записанную числом. Возраст, стоимость билета и класс пассажира — это одинаковые по смыслу числа?

## Поля таблицы

PassengerId — идентификатор (индекс); Survived — 0/1; Pclass — класс 1/2/3; Name — имя; Sex — пол; Age — возраст в годах; SibSp — братья/сёстры и супруги; Parch — родители и дети; Ticket — билет; Fare — стоимость билета; Cabin — каюта; Embarked — порт. Источник и ограничения: `course_support/data/TITANIC_SOURCE.md`.

Это учебная выборка. По ней нельзя автоматически делать выводы обо всех пассажирах и тем более о людях вообще.

## Практический запас: можно ли усреднить две доли? — 12–15 минут

Посчитайте общую долю выживших двумя способами: `df["Survived"].mean()` и среднее двух долей из survival_by_sex. Они совпадают?

Проверьте численность групп. Получите правильную общую долю через сумму числа выживших, делённую на общее число пассажиров. Напишите, почему простое среднее двух долей отвечает другому вопросу. Подсказка: группы имеют разный размер.

Необязательное упражнение, без дополнительных баллов.

In [ ]:
# ✏️ ваш код здесь

### Ещё 5–8 минут: редактор вывода

Обменяйтесь одним выводом из исследования. Найдите в нём: число, группу, единицу измерения и ограничение. Перепишите слишком сильную формулировку так, чтобы она следовала из таблицы. Результат — версия «было/стало»; можно выполнить самостоятельно.

## ЛР1 — самостоятельная работа

Условия: `docs/ЛР1.md` в папке проекта или курса. Создайте отдельную тетрадь в `notebooks`; упражнения сегодня — подготовка, не дополнительная ЛР. Тему проекта обсудим отдельно: для ЛР1 свои данные не требуются. Срок преподаватель объявляет при выдаче. Сдача файлом/архивом или по желанию ссылкой.

## Полезное и интересное

- Уэс Маккинни [«Python for Data Analysis»](https://wesmckinney.com/book/) — книга создателя pandas, бесплатно на сайте автора

Больше — в `docs/ПОЛЕЗНОЕ.md`.

### По желанию после занятия

- [Moneyball](https://www.sonypictures.com/movies/moneyball) — повод подумать о выборе показателя, а не обязательный просмотр.
- [Python for Data Analysis](https://wesmckinney.com/book/), главы 5–6: найдите один приём индексирования или загрузки CSV и попробуйте на нашей таблице. Достаточно 10 минут, не всей главы.